In [1]:
#!pip install datasets bitsandbytes trl
!pip install transformers==4.56.1 peft==0.17.0 accelerate==1.10.0 trl==0.23.1 bitsandbytes==0.47.0 datasets==4.0.0 huggingface-hub==0.34.4 safetensors==0.6.2 pandas==2.2.2 matplotlib==3.10.0 numpy==2.0.2
!pip install flash-linear-attention==0.3.0
!pip install 'transformers>=4.48.0'

In [2]:
import os
import torch
from datasets import Dataset
import re
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import Trainer, DataCollatorForLanguageModeling, EarlyStoppingCallback
from contextlib import nullcontext
from trl import SFTConfig, SFTTrainer
from transformers import set_seed

set_seed(42)

In [5]:
bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
)
repo_id = 'fla-hub/rwkv7-0.4B-world'
model = AutoModelForCausalLM.from_pretrained(repo_id, trust_remote_code=True, quantization_config=bnb_config)
model.config.use_cache = False
model.config.return_dict = True
tokenizer = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)

config.json: 0.00B [00:00, ?B/s]

modeling_rwkv7.py:   0%|          | 0.00/157 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/fla-hub/rwkv7-0.4B-world:
- modeling_rwkv7.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/902M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

...

hf_rwkv_tokenizer.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/fla-hub/rwkv7-0.4B-world:
- hf_rwkv_tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


rwkv_vocab_v20230424.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/45.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [6]:
with open("quatrains_fixed.txt", 'r', encoding='utf-8') as f:
    content = f.read()

poems = re.findall(r'<\|startoftext\|>(.*?)<\|endoftext\|>', content, re.DOTALL)

poems = [p.strip() for p in poems]

dataset_dict = {"text": poems}
dataset = Dataset.from_dict(dataset_dict)

def tokenize_fn(examples):
    # We tokenize the text
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    # For Causal LM, labels are usually a copy of input_ids
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"] # Remove 'text' so only 'input_ids' and 'labels' remain
)

print(dataset)

Parameter 'function'=<function tokenize_fn at 0x7db77fd99940> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/462 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 462
})


In [7]:
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    # the rank of the adapter, the lower the fewer parameters you'll need to train
    r=8,
    lora_alpha=16, # multiplier, usually 2*r
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    # Newer models, such as Phi-3 at time of writing, may require
    # manually setting target modules
    target_modules=['o_proj', 'v_proj', 'r_proj', 'k_proj'],
)
model = get_peft_model(model, config)

In [8]:
print(model.get_memory_footprint()/1e6)

703.082496


In [9]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): RWKV7ForCausalLM(
      (model): RWKV7Model(
        (embeddings): Embedding(65536, 1024)
        (layers): ModuleList(
          (0): RWKV7Block(
            (pre_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attn_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attn): RWKV7Attention(
              (time_shift): ZeroPad2d((0, 0, 1, -1))
              (r_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1024, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1024, bias=False)
                )
              

In [10]:
sft_config = SFTConfig(
    ## GROUP 1: Memory usage
    # These arguments will squeeze the most out of your GPU's RAM
    # Checkpointing
    gradient_checkpointing=True,    # this saves a LOT of memory
    # Set this to avoid exceptions in newer versions of PyTorch
    gradient_checkpointing_kwargs={'use_reentrant': False},
    # Gradient Accumulation / Batch size
    # Actual batch (for updating) is same (1x) as micro-batch size
    gradient_accumulation_steps=2,
    # The initial (micro) batch size to start off with
    per_device_train_batch_size=8,
    # If batch size would cause OOM, halves its size until it works
    auto_find_batch_size=True,

    ## GROUP 2: Dataset-related
    dataset_text_field="text",
    max_length=128, # renamed in v0.20
    # Dataset
    packing=False,
    # packing_strategy='wrapped',

    ## GROUP 3: These are typical training parameters
    num_train_epochs=20,
    warmup_steps=100,
    max_steps=1140,
    learning_rate=5e-5,
    # Optimizer
    # 8-bit Adam optimizer - doesn't help much if you're using LoRA!
    optim='paged_adamw_8bit',
    weight_decay=0.01,
    max_grad_norm=1.0,

    ## GROUP 4: Logging parameters
    logging_steps=10,
    logging_dir='./logs',
    output_dir='./quatrain_rnn',
    report_to='none',

    dataset_kwargs={
        "skip_prepare_dataset": True, # This often bypasses the entropy check
    },

    # ensures bf16 (the new default) is only used when it is actually available
    bf16=torch.cuda.is_bf16_supported(including_emulation=False)
)

In [11]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

early_stopping = EarlyStoppingCallback(early_stopping_patience=5)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=dataset,
)

In [12]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 65530, 'bos_token_id': 0, 'pad_token_id': 0}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: RuntimeWarning: ChunkDeltaRuleFunction does not support float32 on some platforms. Please use bfloat16/float16.
            If you want to use float32, please solve the issue by yourself.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: RuntimeWarning: ChunkDeltaRuleFunction does not support float32 on some platforms. Please use bfloat16/float16.
            If you want to use float32, please solve the issue by yourself.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: RuntimeWarning: ChunkDeltaRuleFunction does not support flo

Step,Training Loss
10,8.332700
20,8.203000
30,8.191600
40,8.181400
50,8.256400
60,7.815400
70,7.882600
80,7.896400
90,7.598000
100,7.457600


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: RuntimeWarning: ChunkDeltaRuleFunction does not support float32 on some platforms. Please use bfloat16/float16.
            If you want to use float32, please solve the issue by yourself.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: RuntimeWarning: ChunkDeltaRuleFunction does not support float32 on some platforms. Please use bfloat16/float16.
            If you want to use float32, please solve the issue by yourself.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: RuntimeWarning: ChunkDeltaRuleFunction does not support float32 on some platforms. Please use bfloat16/float16.
            If you want to use float32, please solve the issue by yourself.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: RuntimeWarning: ChunkDeltaRuleFunction does not suppo

Step,Training Loss
10,8.332700
20,8.203000
30,8.191600
40,8.181400
50,8.256400
60,7.815400
70,7.882600
80,7.896400
90,7.598000
100,7.457600


TrainOutput(global_step=1140, training_loss=5.726241687306187, metrics={'train_runtime': 3509.4166, 'train_samples_per_second': 5.197, 'train_steps_per_second': 0.325, 'total_flos': 5373374407114752.0, 'train_loss': 5.726241687306187, 'epoch': 39.310344827586206})

In [15]:
def generate(model, tokenizer, prompt, max_new_tokens=128, skip_special_tokens=False):
    tokenized_input = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).to(model.device)

    model.eval()
    # if it was trained using mixed precision, uses autocast context
    ctx = torch.autocast(device_type=model.device.type, dtype=model.dtype) \
          if model.dtype in [torch.float16, torch.bfloat16] else nullcontext()
    with ctx:
        gen_output = model.generate(**tokenized_input,
                                    use_cache=False,
                                    eos_token_id=tokenizer.eos_token_id,
                                    pad_token_id=tokenizer.pad_token_id,
                                    max_new_tokens=max_new_tokens)

    output = tokenizer.batch_decode(gen_output, skip_special_tokens=skip_special_tokens)
    return output[0]

In [18]:

prompts = [
            "The solemn bells did mourn the dying day\n",
            "Beneath the frost, the waking river sighed\n",
            "The war-worn king laid down his crown of ash\n",
            "Through autumn’s gold, her silent footsteps came\n",
            "Thy beauty mocks the lilies of the field\n",

          ]


for prompt in prompts:
  print(generate(model, tokenizer, prompt))
  print("----------")
  print()

/usr/local/lib/python3.12/dist-packages/fla/ops/rwkv7/fused_recurrent.py:302: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (9) < num_heads (16). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/fla/ops/rwkv7/fused_recurrent.py:302: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (10) < num_heads (16). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/fla/ops/rwkv7/fused_recurrent.py:302: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (11) < num_heads (16). This may indicate the inputs were passed in 

The solemn bells did mourn the dying day
That in the summer's night was spent;
But in the winter's night the winter's day,
The winter's night was so long and so deep,
That all the winter's night was so long and so deep,
That all the winter's night was so long and so deep,
That all the winter's night was so long and so deep,
And all the winter's night was so long and so deep,
And all the winter's night was so long and so deep,
And all the winter's night was so long and so deep,
And all the winter's night was so long and
----------

Beneath the frost, the waking river sighed
And slumber'd in the summer's heat alone;
But winter's snows came on, and winter's flowers
Fell down, and winter's sweet-smelling rose.
So, in the summer's heat, the waking stream
Was busy in his own sweet self alone;
But, in the summer's heat, the busy stream,
Being in some, was still in others found.
Thus, in the summer's heat, the waking stream
Busied himself in his own sweet self alone;
But, in the summer's heat,